In [ ]:
!pip install --upgrade pip setuptools wheel

In [ ]:
!pip install -q datasets seqeval pytorch-crf

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import random
import numpy as np
import torch
from datasets import load_from_disk

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PATH = "/content/drive/MyDrive/indic-ner/naamapadam_hi_100k"
LOCAL_PATH = "/content/naamapadam_hi_100k"

if os.path.exists(DRIVE_PATH):
    processed = load_from_disk(DRIVE_PATH)
    print("Loaded from Drive")
elif os.path.exists(LOCAL_PATH):
    processed = load_from_disk(LOCAL_PATH)
    print("Loaded from local /content")
else:
    raise FileNotFoundError("Run Phase 1 notebook first and back up to Drive.")

print(processed)

label_names = processed["train"].features["ner_tags"].feature.names
num_labels = len(label_names)
print(label_names)

Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded from Drive
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 100000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 13460
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 867
    })
})
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [ ]:
from collections import Counter

PAD, UNK = "<pad>", "<unk>"
MIN_FREQ = 2

word_counts = Counter(
    token
    for tokens in processed["train"]["tokens"]
    for token in tokens
)

vocab = {PAD: 0, UNK: 1}
for word, count in word_counts.items():
    if count >= MIN_FREQ:
        vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab):,} (min_freq={MIN_FREQ})")

def encode_tokens(tokens: list[str]) -> list[int]:
    return [vocab.get(t, vocab[UNK]) for t in tokens]

def encode_tags(tags: list[int]) -> list[int]:
    return list(tags)

Vocab size: 38,317 (min_freq=2)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class NerDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int):
        row = self.data[idx]
        return {
            "words": torch.tensor(encode_tokens(row["tokens"]), dtype=torch.long),
            "tags": torch.tensor(encode_tags(row["ner_tags"]), dtype=torch.long),
        }

def collate_fn(batch: list[dict]) -> dict:
    max_len = max(len(item["words"]) for item in batch)
    words = torch.zeros(len(batch), max_len, dtype=torch.long)
    tags = torch.zeros(len(batch), max_len, dtype=torch.long)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["words"])
        words[i, :n] = item["words"]
        tags[i, :n] = item["tags"]
        mask[i, :n] = True
    return {"words": words, "tags": tags, "mask": mask}

BATCH_SIZE = 64

train_loader = DataLoader(
    NerDataset(processed["train"]),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    NerDataset(processed["validation"]),
    batch_size=256,
    shuffle=False,
    collate_fn=collate_fn,
)

In [ ]:
import torch.nn as nn
from torchcrf import CRF

class BiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        emb_dim: int = 300,
        hidden_dim: int = 256,
        num_layers: int = 1,
        dropout: float = 0.33,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim // 2,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, words, tags=None, mask=None):
        x = self.dropout(self.embedding(words))
        x, _ = self.lstm(x)
        emissions = self.fc(self.dropout(x))
        if tags is not None:
            return -self.crf(emissions, tags, mask=mask, reduction="mean")
        return self.crf.decode(emissions, mask=mask)

model = BiLSTMCRF(len(vocab), num_labels).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

BiLSTMCRF(
  (embedding): Embedding(38317, 300, padding_idx=0)
  (lstm): LSTM(300, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.33, inplace=False)
  (fc): Linear(in_features=256, out_features=7, bias=True)
  (crf): CRF(num_tags=7)
)
Parameters: 11,937,282


In [ ]:
from seqeval.metrics import classification_report, f1_score

def ids_to_labels(id_seqs: list[list[int]]) -> list[list[str]]:
    return [[label_names[t] for t in seq] for seq in id_seqs]

@torch.no_grad()
def evaluate(loader) -> float:
    model.eval()
    all_true, all_pred = [], []
    for batch in loader:
        words = batch["words"].to(device)
        mask = batch["mask"].to(device)
        preds = model(words, mask=mask)
        for i, pred in enumerate(preds):
            length = int(mask[i].sum().item())
            all_pred.append(pred[:length])
            all_true.append(batch["tags"][i, :length].tolist())
    y_true = ids_to_labels(all_true)
    y_pred = ids_to_labels(all_pred)
    return f1_score(y_true, y_pred), classification_report(y_true, y_pred)

In [ ]:
EPOCHS = 8

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1
)

best_f1 = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        words = batch["words"].to(device)
        tags = batch["tags"].to(device)
        mask = batch["mask"].to(device)
        loss = model(words, tags=tags, mask=mask)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()

    val_f1, report = evaluate(val_loader)
    scheduler.step(val_f1)
    avg_loss = total_loss / len(train_loader)
    history.append({"epoch": epoch, "loss": avg_loss, "val_f1": val_f1})
    print(f"Epoch {epoch:02d} | loss {avg_loss:.4f} | val entity-F1 {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(
            {"model_state": model.state_dict(), "vocab": vocab, "label_names": label_names},
            "/content/bilstm_crf_hi_best.pt",
        )

print(f"\nBest val F1: {best_f1:.4f}")

Epoch 01 | loss 7.2690 | val entity-F1 0.6613
Epoch 02 | loss 4.3499 | val entity-F1 0.6980
Epoch 03 | loss 3.6771 | val entity-F1 0.7184
Epoch 04 | loss 3.3280 | val entity-F1 0.7251
Epoch 05 | loss 3.0997 | val entity-F1 0.7312
Epoch 06 | loss 2.9427 | val entity-F1 0.7313
Epoch 07 | loss 2.8276 | val entity-F1 0.7381
Epoch 08 | loss 2.7219 | val entity-F1 0.7406

Best val F1: 0.7406


In [ ]:
ckpt = torch.load("/content/bilstm_crf_hi_best.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])

val_f1, report = evaluate(val_loader)
print(f"Best-checkpoint val entity-F1: {val_f1:.4f}\n")
print(report)

Best-checkpoint val entity-F1: 0.7406

              precision    recall  f1-score   support

         LOC       0.79      0.80      0.80     10213
         ORG       0.67      0.60      0.63      9786
         PER       0.77      0.79      0.78     10568

   micro avg       0.75      0.73      0.74     30567
   macro avg       0.74      0.73      0.74     30567
weighted avg       0.75      0.73      0.74     30567



In [ ]:
!cp /content/bilstm_crf_hi_best.pt "/content/drive/MyDrive/indic-ner/bilstm_crf_hi_best.pt"
print("Checkpoint saved to Drive: MyDrive/indic-ner/bilstm_crf_hi_best.pt")

Checkpoint saved to Drive: MyDrive/indic-ner/bilstm_crf_hi_best.pt
